# Trabalho 1 - Recuperação da Informação

Grupo:

- Arthur Trottmann Ramos (14681052)
- Maicon Chaves Marques (14593530)

## Instalação de Dependências e Carregamento de Dataset

In [51]:
pip install NLTK numpy pandas ir_datasets

In [52]:
import ir_datasets

dataset = ir_datasets.load("cranfield")

## 1- Pré-Processamento

In [53]:
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, RegexpTokenizer
from nltk.stem import PorterStemmer

nltk.download('stopwords')

stemmer = PorterStemmer()


def tokenization(text):
  tokenizer = RegexpTokenizer(r'\w+')
  clean_tokens = tokenizer.tokenize(text)
  return lower_case_normalization(clean_tokens)

def remove_stopwords(words):
  stopwords_set = set(stopwords.words('english'))
  filtered_words = [word for word in words if word not in stopwords_set]
  return filtered_words

def lower_case_normalization(words):
  normalized_words = [word.lower() for word in words]
  return normalized_words

def stemming(words):
  stemmed_words = [stemmer.stem(word) for word in words]
  return stemmed_words

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [54]:
def preprocess(text, config_type=0):
  words = tokenization(text)

  if config_type == 1:
    words = remove_stopwords(words)
  if config_type == 2:
    words = stemming(words)
  if config_type == 3:
    words = remove_stopwords(words)
    words = stemming(words)

  return words

## Índice Invertido

In [55]:
class InvertedIndex:
    def __init__(self):
        self.index = {}
        self.document_length = {}
        self.n_documents = 0
        self.avgdl = 0.0

    def build(self, documents, preprocessing_type=0):
        for doc in documents:
            doc_id = doc[0]
            doc_title = doc[1]
            doc_text = doc[2]
            doc_author = doc[3]

            self.n_documents += 1

            title_words = preprocess(doc_title, preprocessing_type)
            text_words = preprocess(doc_text, preprocessing_type)
            author_words = preprocess(doc_author, preprocessing_type)

            all_words = title_words + text_words + author_words

            self.document_length[doc_id] = len(all_words)

            for word in all_words:
                if word not in self.index:
                    self.index[word] = {}

                if doc_id not in self.index[word]:
                    self.index[word][doc_id] = 0

                self.index[word][doc_id] += 1

        self.avgdl = (
            sum(self.document_length.values())
            / len(self.document_length)
        )

## 2- Modelo Vetorial

In [56]:
from collections import Counter

class ModeloVetorial:
    def __init__(self, inverted_index):
        self.inverted_index = inverted_index

        # Matriz de pesos: weights[termo][documento] = peso TF-IDF
        self.weights = {}

        # Norma de cada vetor de documento
        self.document_norm = {}

        N = self.inverted_index.n_documents

        for word in self.inverted_index.index:
            self.weights[word] = {}

            ni = len(self.inverted_index.index[word])
            idf = math.log(N / ni)

            for doc_id, fi_j in self.inverted_index.index[word].items():
                wi_j = (1 + math.log(fi_j)) * idf

                # Guarda o peso TF-IDF na matriz
                self.weights[word][doc_id] = wi_j

                if doc_id not in self.document_norm:
                    self.document_norm[doc_id] = 0.0

                self.document_norm[doc_id] += wi_j ** 2

        # Finaliza o cálculo da norma dos documentos
        for doc_id in self.document_norm:
            self.document_norm[doc_id] = math.sqrt(
                self.document_norm[doc_id]
            )

    def score(self, query, doc_id, preprocessing_type=0):
        query_words = preprocess(query, preprocessing_type)
        query_frequency = Counter(query_words)

        query_weights = {}
        product = 0.0
        query_norm = 0.0

        N = self.inverted_index.n_documents

        for word, fi_q in query_frequency.items():
            if word not in self.inverted_index.index:
                continue

            ni = len(self.inverted_index.index[word])
            idf = math.log(N / ni)

            wi_q = (1 + math.log(fi_q)) * idf

            # Guarda o peso TF-IDF do termo na consulta
            query_weights[word] = wi_q

            query_norm += wi_q ** 2

            # Busca o peso do documento diretamente na matriz
            wi_j = self.weights[word].get(doc_id, 0.0)

            product += wi_j * wi_q

        query_norm = math.sqrt(query_norm)
        document_norm = self.document_norm.get(doc_id, 0.0)

        if query_norm == 0 or document_norm == 0:
            return 0.0

        similarity = product / (document_norm * query_norm)

        return similarity

## 3- Modelo Probabilístico (BM25)

In [57]:
import math

class BM25:
    def __init__(self, inverted_index, k1=0.5, b=0):
        self.inverted_index = inverted_index
        self.k1 = k1
        self.b = b

    def score(self, query, doc_id, preprocessing_type=0):
        score = 0.0
        query_words = preprocess(query, preprocessing_type)

        for word in query_words:
            if word in self.inverted_index.index and doc_id in self.inverted_index.index[word]:
                tf = self.inverted_index.index[word][doc_id]
                df = len(self.inverted_index.index[word])
                idf = math.log(1 + ((self.inverted_index.n_documents - df + 0.5) / (df + 0.5)))
                dl = self.inverted_index.document_length[doc_id]
                avgdl = self.inverted_index.avgdl
                score += idf * ((tf * (self.k1 + 1)) / (tf + self.k1 * (1 - self.b + self.b * (dl / avgdl))))

        return score

## 4- Métricas de Avaliação

In [58]:
def precision_at_k(retrieved_docs, relevant_docs, k):
    retrieved_k = retrieved_docs[:k]
    relevant_retrieved = [doc for doc in retrieved_k if doc in relevant_docs]
    precision = len(relevant_retrieved) / k
    return precision

def recall_at_k(retrieved_docs, relevant_docs, k):
    retrieved_k = retrieved_docs[:k]
    relevant_retrieved = [doc for doc in retrieved_k if doc in relevant_docs]
    recall = len(relevant_retrieved) / len(relevant_docs) if relevant_docs else 0
    return recall

def AP(retrieved_docs, relevant_docs):
    relevant_retrieved = [doc for doc in retrieved_docs if doc in relevant_docs]
    if not relevant_retrieved:
        return 0.0

    precision_sum = 0.0
    for i, doc in enumerate(retrieved_docs):
        if doc in relevant_docs:
            precision_sum += precision_at_k(retrieved_docs, relevant_docs, i + 1)

    average_precision = precision_sum / len(relevant_retrieved)
    return average_precision

## Rodando Modelos

In [59]:
# Para cada query, armazena em sets os documentos relevantes (grau de relevância >= 1) de qrels

relevant_docs_per_query = {}

for qrel in dataset.qrels_iter():
    query_id = qrel[0]
    doc_id = qrel[1]
    relevance_grade = qrel[2]

    if relevance_grade >= 1:
        if query_id not in relevant_docs_per_query:
            relevant_docs_per_query[query_id] = set()
        relevant_docs_per_query[query_id].add(doc_id)

In [60]:
preprocessing_type = 0
k = 10

inverted_index = InvertedIndex()

inverted_index.build(
    dataset.docs_iter(),
    preprocessing_type=preprocessing_type
)

bm25 = BM25(inverted_index, 0.5, 0)

modelo_vetorial = ModeloVetorial(inverted_index)

metrics_bm25 = {}
metrics_vetorial = {}

for query in dataset.queries_iter():
    query_id = query[0]
    query_text = query[1]

    scores_bm25 = {}
    scores_vetorial = {}

    for doc in dataset.docs_iter():
        doc_id = doc[0]

        score_bm25 = bm25.score(
            query_text,
            doc_id,
            preprocessing_type=preprocessing_type
        )

        scores_bm25[doc_id] = score_bm25

        score_vetorial = modelo_vetorial.score(
            query_text,
            doc_id,
            preprocessing_type=preprocessing_type
        )

        scores_vetorial[doc_id] = score_vetorial

    retrieved_docs_bm25 = sorted(
        scores_bm25,
        key=scores_bm25.get,
        reverse=True
    )

    retrieved_docs_vetorial = sorted(
        scores_vetorial,
        key=scores_vetorial.get,
        reverse=True
    )

    metrics_bm25[query_id] = [
        precision_at_k(
            retrieved_docs_bm25,
            relevant_docs_per_query.get(query_id, set()),
            k
        ),
        recall_at_k(
            retrieved_docs_bm25,
            relevant_docs_per_query.get(query_id, set()),
            k
        ),
        AP(
            retrieved_docs_bm25,
            relevant_docs_per_query.get(query_id, set())
        )
    ]

    metrics_vetorial[query_id] = [
        precision_at_k(
            retrieved_docs_vetorial,
            relevant_docs_per_query.get(query_id, set()),
            k
        ),
        recall_at_k(
            retrieved_docs_vetorial,
            relevant_docs_per_query.get(query_id, set()),
            k
        ),
        AP(
            retrieved_docs_vetorial,
            relevant_docs_per_query.get(query_id, set())
        )
    ]

In [61]:
print("Métricas do BM25:")
print(metrics_bm25)

print("\nMétricas do Modelo Vetorial:")
print(metrics_vetorial)

Métricas do BM25:
{'1': [0.5, 0.17857142857142858, 0.16703864206536617], '2': [0.3, 0.125, 0.15403677530626292], '3': [0.4, 0.5, 0.482742467644968], '4': [0.1, 0.5, 0.5434782608695652], '5': [0.2, 0.5, 0.20665584415584418], '6': [0.1, 0.25, 0.14848822513578958], '7': [0.2, 0.4, 0.19596276698578488], '8': [0.1, 0.09090909090909091, 0.13616026347717042], '9': [0.3, 1.0, 0.6095238095238095], '10': [0.1, 0.125, 0.0688816805435756], '11': [0.1, 0.14285714285714285, 0.12089282968659408], '12': [0.2, 0.4, 0.1801906793109139], '13': [0.0, 0.0, 0.002910396107870574], '14': [0.1, 0.5, 0.5588235294117647], '15': [0.2, 1.0, 1.0], '16': [0.1, 0.3333333333333333, 0.1488095238095238], '17': [0.0, 0.0, 0.029351395730706074], '18': [0.1, 0.3333333333333333, 0.20744241534367258], '19': [0.0, 0.0, 0.029120301678413132], '20': [0.4, 0.4444444444444444, 0.3061552719853373], '21': [0.0, 0.0, 0.05015207085668364], '22': [0.0, 0.0, 0.0015337423312883436], '23': [0.1, 0.03125, 0.1112333654289845], '24': [0.2, 

## 5- Comparação entre modelos

In [62]:
#Rodar os modelos e avaliar em quais consultas ocorreram as maiores divergências e hipóteses do porquê (Arthur)

## 6- Análise por consulta

In [63]:
#Identificar 2 consultas que BM25 > vetorial, 2 em que vetorial > BM25 e 2 em que ambos foram ruins (Arthur)

## 7- Variação dos parâmetros do BM25

In [64]:
#Impacto na mudança de parâmetros do BM25 (Arthur)

## 8- Modificação de consultas

In [65]:
# @title
import pandas as pd
from IPython.display import display

# Normaliza os IDs porque algumas versões do Cranfield usam
# preenchimento com zeros, como "057", enquanto outras usam "57".
todas_as_consultas = {
    str(int(str(query[0]).strip())): {
        "id_dataset": query[0],
        "texto": query[1]
    }
    for query in dataset.queries_iter()
}

print(f"Total de consultas no Cranfield: {len(todas_as_consultas)}")


# Guarda os dados dos documentos para evitar percorrer o dataset
# repetidamente durante a exibição dos resultados.
documentos_cranfield = {
    doc[0]: {
        "titulo": doc[1],
        "texto": doc[2],
        "autores": doc[3]
    }
    for doc in dataset.docs_iter()
}


pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 180)

def resumir_campo(valor, limite):
    """Remove quebras de linha e abrevia um campo para exibição."""
    valor = " ".join(str(valor).split())

    if len(valor) <= limite:
        return valor

    return valor[:limite - 3] + "..."


def gerar_top10(texto_consulta, modelo, query_id):
    """
    Calcula o ranking de uma consulta e devolve os dez primeiros
    documentos em um DataFrame.
    """
    pontuacoes = {
        doc_id: modelo.score(
            texto_consulta,
            doc_id,
            preprocessing_type=preprocessing_type
        )
        for doc_id in documentos_cranfield
    }

    ranking = sorted(
        pontuacoes.items(),
        key=lambda item: (-item[1], int(item[0]))
    )[:10]

    # O julgamento de relevância continua associado ao par:
    # (query_id, doc_id).
    id_dataset = todas_as_consultas[query_id]["id_dataset"]

    relevantes = (
        relevant_docs_per_query.get(id_dataset, set())
        | relevant_docs_per_query.get(query_id, set())
    )

    linhas = [
        {
            "Posição": posicao,
            "Doc.": doc_id,
            "Score": round(score, 6),
            "Relevante": "Sim" if doc_id in relevantes else "Não",
            "Título": resumir_campo(
                documentos_cranfield[doc_id]["titulo"],
                100
            ),
            "Autores": resumir_campo(
                documentos_cranfield[doc_id]["autores"],
                70
            ),
            "Texto": resumir_campo(
                documentos_cranfield[doc_id]["texto"],
                240
            )
        }
        for posicao, (doc_id, score) in enumerate(
            ranking,
            start=1
        )
    ]

    return pd.DataFrame(linhas)


def exibir_mudancas_top10(
    top10_original,
    top10_alternativa,
    nome_modelo
):
    """
    Mostra quais documentos permaneceram, entraram ou saíram
    do Top-10 após a modificação da consulta.
    """
    docs_original = top10_original["Doc."].tolist()
    docs_alternativa = top10_alternativa["Doc."].tolist()

    permaneceram = [
        doc_id
        for doc_id in docs_alternativa
        if doc_id in docs_original
    ]

    entraram = [
        doc_id
        for doc_id in docs_alternativa
        if doc_id not in docs_original
    ]

    sairam = [
        doc_id
        for doc_id in docs_original
        if doc_id not in docs_alternativa
    ]

    print(f"\nMudanças no Top-10 — {nome_modelo}")
    print(f"Documentos mantidos: {permaneceram}")
    print(f"Documentos que entraram: {entraram}")
    print(f"Documentos que saíram: {sairam}")
    print(f"Sobreposição: {len(permaneceram)}/10")


def exibir_comparacao(
    numero,
    query_id,
    consulta_original,
    consulta_alternativa,
    tipo_modificacao,
    detalhes_modificacao
):
    """
    Executa a consulta original e a alternativa nos dois modelos,
    exibindo as modificações, os rankings e as mudanças no Top-10.
    """
    if query_id not in todas_as_consultas:
        raise ValueError(
            f"A consulta {query_id} não existe no Cranfield."
        )

    print("\n" + "=" * 120)
    print(f"CONSULTA {numero} | ID Cranfield: {query_id}")
    print(f"Modificações utilizadas: {tipo_modificacao}")

    print("\nDETALHES DAS MODIFICAÇÕES:")
    for detalhe in detalhes_modificacao:
        print(f"- {detalhe}")

    print("\nCONSULTA ORIGINAL:")
    print(consulta_original)

    print("\nCONSULTA ALTERNATIVA:")
    print(consulta_alternativa)

    # Modelo Vetorial
    top10_vetorial_original = gerar_top10(
        consulta_original,
        modelo_vetorial,
        query_id
    )

    top10_vetorial_alternativa = gerar_top10(
        consulta_alternativa,
        modelo_vetorial,
        query_id
    )

    print("\nMODELO VETORIAL — CONSULTA ORIGINAL")
    display(top10_vetorial_original)

    print("\nMODELO VETORIAL — CONSULTA ALTERNATIVA")
    display(top10_vetorial_alternativa)

    exibir_mudancas_top10(
        top10_vetorial_original,
        top10_vetorial_alternativa,
        "Modelo Vetorial"
    )

    # BM25
    top10_bm25_original = gerar_top10(
        consulta_original,
        bm25,
        query_id
    )

    top10_bm25_alternativa = gerar_top10(
        consulta_alternativa,
        bm25,
        query_id
    )

    print("\nBM25 — CONSULTA ORIGINAL")
    display(top10_bm25_original)

    print("\nBM25 — CONSULTA ALTERNATIVA")
    display(top10_bm25_alternativa)

    exibir_mudancas_top10(
        top10_bm25_original,
        top10_bm25_alternativa,
        "BM25"
    )


# ============================================================
# CONSULTA 1
# ============================================================

consulta_1_id = "57"
consulta_1_original = todas_as_consultas[consulta_1_id]["texto"]

consulta_1_alternativa = (
    "what steady and non-steady aerodynamic flow characteristics "
    "affect the wing flutter mechanism?"
)

exibir_comparacao(
    numero=1,
    query_id=consulta_1_id,
    consulta_original=consulta_1_original,
    consulta_alternativa=consulta_1_alternativa,
    tipo_modificacao=(
        "Remover termos + acrescentar termos + tornar mais específica"
    ),
    detalhes_modificacao=[
        (
            "Remover termos: foi removido o termo genérico "
            "'significant'."
        ),
        (
            "Acrescentar termos: foram acrescentados "
            "'aerodynamic' e 'wing'."
        ),
        (
            "Tornar mais específica: a consulta passou a relacionar "
            "o escoamento aerodinâmico ao flutter de asas."
        )
    ]
)


# ============================================================
# CONSULTA 2
# ============================================================

consulta_2_id = "121"
consulta_2_original = todas_as_consultas[consulta_2_id]["texto"]

consulta_2_alternativa = (
    "what papers deal with circumferential buckling of thin cylindrical "
    "shells due to thermal or mechanical loading?"
)

exibir_comparacao(
    numero=2,
    query_id=consulta_2_id,
    consulta_original=consulta_2_original,
    consulta_alternativa=consulta_2_alternativa,
    tipo_modificacao=(
        "Remover termos + acrescentar termos + tornar mais específica"
    ),
    detalhes_modificacao=[
        (
            "Remover termos: foram removidos 'are there' "
            "e a repetição de 'buckling'."
        ),
        (
            "Acrescentar termos: foi acrescentada a expressão "
            "'thin cylindrical shells'."
        ),
        (
            "Tornar mais específica: a consulta foi delimitada "
            "à flambagem de cascas cilíndricas finas."
        )
    ]
)


# ============================================================
# CONSULTA 3
# ============================================================

consulta_3_id = "183"
consulta_3_original = todas_as_consultas[consulta_3_id]["texto"]

consulta_3_alternativa = (
    "what factors, including lift and aircraft geometry, have been shown "
    "to have a primary influence on sonic boom intensity?"
)

exibir_comparacao(
    numero=3,
    query_id=consulta_3_id,
    consulta_original=consulta_3_original,
    consulta_alternativa=consulta_3_alternativa,
    tipo_modificacao=(
        "Acrescentar termos + usar sinônimos + tornar mais específica"
    ),
    detalhes_modificacao=[
        (
            "Acrescentar termos: foram acrescentados "
            "'lift' e 'aircraft geometry'."
        ),
        (
            "Usar sinônimos: 'strength' foi substituído "
            "por 'intensity'."
        ),
        (
            "Tornar mais específica: a consulta passou a mencionar "
            "fatores aerodinâmicos relacionados ao sonic boom."
        )
    ]
)


# ============================================================
# CONSULTA 4
# ============================================================

consulta_4_id = "56"
consulta_4_original = todas_as_consultas[consulta_4_id]["texto"]

consulta_4_alternativa = (
    "to what extent can steady-state aerodynamic data be used to "
    "estimate wing flutter characteristics?"
)

exibir_comparacao(
    numero=4,
    query_id=consulta_4_id,
    consulta_original=consulta_4_original,
    consulta_alternativa=consulta_4_alternativa,
    tipo_modificacao=(
        "Acrescentar termos + usar sinônimos + tornar mais específica"
    ),
    detalhes_modificacao=[
        (
            "Acrescentar termos: foi acrescentado o termo 'wing'."
        ),
        (
            "Usar sinônimos: 'utilized' foi substituído por 'used' "
            "e 'predict' foi substituído por 'estimate'."
        ),
        (
            "Tornar mais específica: a consulta passou a tratar "
            "das características de flutter de asas."
        )
    ]
)


# ============================================================
# CONSULTA 5
# ============================================================

consulta_5_id = "219"
consulta_5_original = todas_as_consultas[consulta_5_id]["texto"]

consulta_5_alternativa = (
    "what are the effects on viscous flow fields around circular "
    "cylinders when the Reynolds number is low?"
)

exibir_comparacao(
    numero=5,
    query_id=consulta_5_id,
    consulta_original=consulta_5_original,
    consulta_alternativa=consulta_5_alternativa,
    tipo_modificacao=(
        "Acrescentar termos + usar sinônimos + tornar mais específica"
    ),
    detalhes_modificacao=[
        (
            "Acrescentar termos: foram acrescentados "
            "'viscous' e 'circular cylinders'."
        ),
        (
            "Usar sinônimos: 'small' foi substituído por 'low'."
        ),
        (
            "Tornar mais específica: a consulta foi delimitada "
            "ao escoamento viscoso ao redor de cilindros circulares."
        )
    ]
)

Total de consultas no Cranfield: 225

CONSULTA 1 | ID Cranfield: 57
Modificações utilizadas: Remover termos + acrescentar termos + tornar mais específica

DETALHES DAS MODIFICAÇÕES:
- Remover termos: foi removido o termo genérico 'significant'.
- Acrescentar termos: foram acrescentados 'aerodynamic' e 'wing'.
- Tornar mais específica: a consulta passou a relacionar o escoamento aerodinâmico ao flutter de asas.

CONSULTA ORIGINAL:
what are the significant steady and non-steady flow characteristics
which affect the flutter mechanism .

CONSULTA ALTERNATIVA:
what steady and non-steady aerodynamic flow characteristics affect the wing flutter mechanism?

MODELO VETORIAL — CONSULTA ORIGINAL


,Posição,Doc.,Score,Relevante,Título,Autores,Texto
0,1,753,0.171582,Não,development of a quasi-steady approach to flutter and correlation with kernel-function...,"gravitz,s.i., laidlaw,w.r., bryce,w.w. and cooper,r.e.",development of a quasi-steady approach to flutter and correlation with kernel-function...
1,2,1099,0.150651,Não,a theoretical study of stagnation point ablation .,"roberts, l.",a theoretical study of stagnation point ablation . a simplified analysis is made of th...
2,3,380,0.135794,Sim,effect of quasi-steady air forces on incompressible bending-torsion flutter .,"dugundi,j.",effect of quasi-steady air forces on incompressible bending-torsion flutter . explicit...
3,4,117,0.115749,Não,the motion of a viscous liquid past a paraboloid .,"mather,d.j.",the motion of a viscous liquid past a paraboloid . an approximate solution for the ste...
4,5,1339,0.114794,Não,calculation of flutter characteristics for finite-span swept or unswept wings at subso...,"yates, e.c.",calculation of flutter characteristics for finite-span swept or unswept wings at subso...
5,6,444,0.114338,Não,an approach to the flutter problem in real fluids .,"rott,n. and george,m.b.t.",an approach to the flutter problem in real fluids . an approximate theory of airfoils ...
6,7,1181,0.113623,Não,steady magnetohydrodynamic flow past a non-conducting wedge .,"chu,c.k. and lynn,y.m.",steady magnetohydrodynamic flow past a non-conducting wedge . this paper presents a st...
7,8,52,0.113242,Não,procedure for calculating flutter at high supersonic speed including camber deflection...,"morgan,h.g.",procedure for calculating flutter at high supersonic speed including camber deflection...
8,9,1111,0.111271,Não,some research on high speed flutter .,"garrick, i.e.",some research on high speed flutter . paper presents brief discussions of many topics ...
9,10,916,0.109796,Não,the flow around oscillating low aspect ratio wings at transonic speeds .,"landahl, m.t.",the flow around oscillating low aspect ratio wings at transonic speeds . when certain ...



MODELO VETORIAL — CONSULTA ALTERNATIVA


,Posição,Doc.,Score,Relevante,Título,Autores,Texto
0,1,753,0.189901,Não,development of a quasi-steady approach to flutter and correlation with kernel-function...,"gravitz,s.i., laidlaw,w.r., bryce,w.w. and cooper,r.e.",development of a quasi-steady approach to flutter and correlation with kernel-function...
1,2,52,0.154801,Não,procedure for calculating flutter at high supersonic speed including camber deflection...,"morgan,h.g.",procedure for calculating flutter at high supersonic speed including camber deflection...
2,3,380,0.136661,Sim,effect of quasi-steady air forces on incompressible bending-torsion flutter .,"dugundi,j.",effect of quasi-steady air forces on incompressible bending-torsion flutter . explicit...
3,4,1339,0.132610,Não,calculation of flutter characteristics for finite-span swept or unswept wings at subso...,"yates, e.c.",calculation of flutter characteristics for finite-span swept or unswept wings at subso...
4,5,1111,0.129737,Não,some research on high speed flutter .,"garrick, i.e.",some research on high speed flutter . paper presents brief discussions of many topics ...
5,6,916,0.127010,Não,the flow around oscillating low aspect ratio wings at transonic speeds .,"landahl, m.t.",the flow around oscillating low aspect ratio wings at transonic speeds . when certain ...
6,7,749,0.125625,Não,the aerodynamic effects of aspect ratio and sweepback on wing flutter .,"molyneux,w.g. and hall,h.",the aerodynamic effects of aspect ratio and sweepback on wing flutter . the report des...
7,8,391,0.118637,Não,flutter of rectangular simply supported panels at high supersonic speeds .,"hedgepeth,j.m.",flutter of rectangular simply supported panels at high supersonic speeds . the problem...
8,9,117,0.116605,Não,the motion of a viscous liquid past a paraboloid .,"mather,d.j.",the motion of a viscous liquid past a paraboloid . an approximate solution for the ste...
9,10,444,0.112582,Não,an approach to the flutter problem in real fluids .,"rott,n. and george,m.b.t.",an approach to the flutter problem in real fluids . an approximate theory of airfoils ...



Mudanças no Top-10 — Modelo Vetorial
Documentos mantidos: ['753', '52', '380', '1339', '1111', '916', '117', '444']
Documentos que entraram: ['749', '391']
Documentos que saíram: ['1099', '1181']
Sobreposição: 8/10

BM25 — CONSULTA ORIGINAL


...


Mudanças no Top-10 — Modelo Vetorial
Documentos mantidos: ['117']
Documentos que entraram: ['1084', '1081', '533', '1105', '1176', '139', '483', '1078', '36']
Documentos que saíram: ['1221', '268', '342', '809', '236', '1222', '993', '456', '828']
Sobreposição: 1/10

BM25 — CONSULTA ORIGINAL


,Posição,Doc.,Score,Relevante,Título,Autores,Texto
0,1,992,13.896530,Não,the effects of a small jet of air exhausting from the nose of a body of revolution in ...,"love, e.s.",the effects of a small jet of air exhausting from the nose of a body of revolution in ...
1,2,1222,13.872676,Não,axisymmetric magnetohydrodynamic channel flow .,"hains,f.d. and holer,y.a.",axisymmetric magnetohydrodynamic channel flow . the axisymmetric subsonic and superson...
2,3,993,13.314147,Não,the extent of the jet interference flow fields . jet effects on cylindrical afterbodie...,"hayman, l.o. and mcdearmon, r.w.",the extent of the jet interference flow fields . jet effects on cylindrical afterbodie...
3,4,329,13.234804,Não,various aerodynamic characteristics in hypersonic rarefied gas flow .,"probstein,r.f. and kemp,n.h.",various aerodynamic characteristics in hypersonic rarefied gas flow . this paper consi...
4,5,456,12.713524,Não,a study of flow fields about some typical blunt-nosed slender bodies .,"vaglio-laurin,r. and trella,m.",a study of flow fields about some typical blunt-nosed slender bodies . complete invisc...
5,6,828,12.436058,Não,stresses and small displacements of shallow spherical shells .,"reissner,e.",stresses and small displacements of shallow spherical shells . the purpose of the pres...
6,7,433,12.123221,Não,application of two dimensional vortex theory to the prediction of flow fields behind w...,"rogers,a.w.",application of two dimensional vortex theory to the prediction of flow fields behind w...
7,8,315,11.969176,Não,scale effects at high subsonic and transonic speeds and methods for fixing transition ...,"haines,a.b., holder,d.w. and pearcey,h.h.",scale effects at high subsonic and transonic speeds and methods for fixing transition ...
8,9,1082,11.961014,Não,"the flow past pitot tube at low reynolds numbers, part 1-dash the numerical solution o...","lester, w.g.s.","the flow past pitot tube at low reynolds numbers, part 1-dash the numerical solution o..."
9,10,236,11.904061,Não,criteria for thermodynamic equilibrium in gas flow .,"rudin,m.","criteria for thermodynamic equilibrium in gas flow . when gases flow at high velocity,..."



BM25 — CONSULTA ALTERNATIVA


,Posição,Doc.,Score,Relevante,Título,Autores,Texto
0,1,1082,16.258755,Não,"the flow past pitot tube at low reynolds numbers, part 1-dash the numerical solution o...","lester, w.g.s.","the flow past pitot tube at low reynolds numbers, part 1-dash the numerical solution o..."
1,2,1313,15.021925,Não,on the flow in a reflected shock tunnel .,"holder, d.w. and schultz, d.l.",on the flow in a reflected shock tunnel . the performance of a shock tunnel operated b...
2,3,1081,14.997212,Sim,numerical solution of the navier-stokes equations for the flow around a circular cylin...,"kawaguti, m.",numerical solution of the navier-stokes equations for the flow around a circular cylin...
3,4,139,14.832066,Não,viscous effects on pitot tubes at low speeds .,"mcmillan,f.a.",viscous effects on pitot tubes at low speeds . measurements were made of the pressure ...
4,5,576,14.173730,Não,viscous and inviscid stagnation flow in a dissociated hypervelocity free stream .,"inger, g.r.",viscous and inviscid stagnation flow in a dissociated hypervelocity free stream . high...
5,6,25,13.655394,Não,inviscid hypersonic flow over blunt-nosed slender bodies .,"lees,l. and kubota,t.",inviscid hypersonic flow over blunt-nosed slender bodies . at hypersonic speeds the dr...
6,7,1084,13.387935,Não,the flow past circular cylinders at low speeds .,"thom, a.",the flow past circular cylinders at low speeds . this paper deals chiefly with calcula...
7,8,1214,13.113261,Sim,the drag of elongated bodies over a wide reynolds number range .,"robertson,j.m. and clark,m.e.",the drag of elongated bodies over a wide reynolds number range . the resistance of bod...
8,9,112,13.010669,Não,steady motion of conducting fluids in pipes under transverse magnetic fields .,"shercliff,j.a.",steady motion of conducting fluids in pipes under transverse magnetic fields . this pa...
9,10,208,12.930001,Não,the hall effect in the viscous flow of ionized gas between parallel plates under trans...,"sato,h.",the hall effect in the viscous flow of ionized gas between parallel plates under trans...



Mudanças no Top-10 — BM25
Documentos mantidos: ['1082']
Documentos que entraram: ['1313', '1081', '139', '576', '25', '1084', '1214', '112', '208']
Documentos que saíram: ['992', '1222', '993', '329', '456', '828', '433', '315', '236']
Sobreposição: 1/10


## 9- Análise de erros

In [66]:
# @title
import math
import pandas as pd
from collections import Counter
from IPython.display import display

QUERIES = ["57"]
TOP_K = 10
TOPO = 5
N_FALSOS_POSITIVOS = 2
N_RELEVANTES_OMITIDOS = 2

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)


def normalizar_id(valor):
    texto = str(valor).strip()
    return str(int(texto)) if texto.isdigit() else texto


def chave_id(valor):
    texto = normalizar_id(valor)
    return (0, int(texto)) if texto.isdigit() else (1, texto)


def resumir(valor, limite):
    texto = " ".join(str(valor).split())
    return texto if len(texto) <= limite else texto[:limite - 3] + "..."


consultas = {
    normalizar_id(query[0]): {
        "id_dataset": query[0],
        "texto": query[1]
    }
    for query in dataset.queries_iter()
}

documentos = {
    normalizar_id(doc[0]): {
        "id_dataset": doc[0],
        "titulo": doc[1],
        "texto": doc[2],
        "autores": doc[3]
    }
    for doc in dataset.docs_iter()
}

relevantes = {}

for qrel in dataset.qrels_iter():
    if qrel[2] >= 1:
        query_id = normalizar_id(qrel[0])
        doc_id = normalizar_id(qrel[1])

        relevantes.setdefault(
            query_id,
            set()
        ).add(doc_id)

modelos = {
    "Modelo Vetorial": modelo_vetorial,
    "BM25": bm25
}

cache_rankings = {}


def calcular_ranking(query_id, nome_modelo):
    query_id = normalizar_id(query_id)
    chave = (query_id, nome_modelo)

    if chave in cache_rankings:
        return cache_rankings[chave]

    texto_query = consultas[query_id]["texto"]
    modelo = modelos[nome_modelo]

    ranking = [
        (
            doc_id,
            modelo.score(
                texto_query,
                dados["id_dataset"],
                preprocessing_type=preprocessing_type
            )
        )
        for doc_id, dados in documentos.items()
    ]

    ranking.sort(
        key=lambda item: (
            -item[1],
            chave_id(item[0])
        )
    )

    cache_rankings[chave] = ranking
    return ranking


def selecionar_erros(query_id, ranking):
    docs_relevantes = relevantes.get(query_id, set())

    falsos_positivos = [
        (posicao, doc_id, score)
        for posicao, (doc_id, score) in enumerate(
            ranking[:TOPO],
            start=1
        )
        if doc_id not in docs_relevantes
    ][:N_FALSOS_POSITIVOS]

    relevantes_omitidos = [
        (posicao, doc_id, score)
        for posicao, (doc_id, score) in enumerate(
            ranking[TOP_K:],
            start=TOP_K + 1
        )
        if doc_id in docs_relevantes
    ][:N_RELEVANTES_OMITIDOS]

    return falsos_positivos, relevantes_omitidos


def montar_tabela(
    query_id,
    itens,
    falsos=None,
    omitidos=None
):
    falsos = set() if falsos is None else falsos
    omitidos = set() if omitidos is None else omitidos

    docs_relevantes = relevantes.get(query_id, set())
    linhas = []

    for posicao, doc_id, score in itens:
        dados = documentos[doc_id]

        if doc_id in falsos:
            categoria = "Falso positivo selecionado"
        elif doc_id in omitidos:
            categoria = "Relevante fora do Top-10"
        else:
            categoria = ""

        linhas.append({
            "Posição": posicao,
            "Doc.": doc_id,
            "Score": score,
            "Relevante": (
                "Sim"
                if doc_id in docs_relevantes
                else "Não"
            ),
            "Categoria": categoria,
            "Título": resumir(
                dados["titulo"],
                100
            ),
            "Texto": resumir(
                dados["texto"],
                260
            )
        })

    return pd.DataFrame(linhas)


def estilo_categoria(linha):
    categoria = linha.get("Categoria", "")

    if categoria == "Falso positivo selecionado":
        estilo = (
            "background-color: #f8d7da;"
            "color: #721c24;"
            "font-weight: bold;"
            "border-top: 2px solid #dc3545;"
            "border-bottom: 2px solid #dc3545;"
        )

        return [estilo] * len(linha)

    if categoria == "Relevante fora do Top-10":
        estilo = (
            "background-color: #d4edda;"
            "color: #155724;"
            "font-weight: bold;"
            "border-top: 2px solid #28a745;"
            "border-bottom: 2px solid #28a745;"
        )

        return [estilo] * len(linha)

    return [""] * len(linha)


def estilizar(tabela, formatos=None):
    resultado = (
        tabela.style
        .apply(
            estilo_categoria,
            axis=1
        )
        .set_properties(**{
            "text-align": "left",
            "vertical-align": "top"
        })
        .set_table_styles([{
            "selector": "th",
            "props": [
                ("background-color", "#343a40"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("text-align", "left")
            ]
        }])
    )

    if formatos:
        resultado = resultado.format(formatos)

    return resultado


def diagnostico_vetorial(
    query_id,
    doc_id,
    categoria
):
    texto_query = consultas[query_id]["texto"]
    doc_dataset_id = documentos[doc_id]["id_dataset"]

    frequencias_query = Counter(
        preprocess(
            texto_query,
            preprocessing_type
        )
    )

    N = inverted_index.n_documents
    produto_escalar = 0.0
    soma_quadrados_query = 0.0
    linhas = []

    for termo, tf_query in frequencias_query.items():
        postings = inverted_index.index.get(
            termo,
            {}
        )

        df = len(postings)

        if df > 0:
            idf = math.log(N / df)

            peso_query = (
                1 + math.log(tf_query)
            ) * idf

            tf_documento = postings.get(
                doc_dataset_id,
                0
            )

            peso_documento = (
                modelo_vetorial
                .weights[termo]
                .get(doc_dataset_id, 0.0)
            )
        else:
            idf = 0.0
            peso_query = 0.0
            tf_documento = 0
            peso_documento = 0.0

        contribuicao = (
            peso_query * peso_documento
        )

        produto_escalar += contribuicao
        soma_quadrados_query += peso_query ** 2

        linhas.append({
            "Categoria": categoria,
            "Doc.": doc_id,
            "Termo": termo,
            "TF query": tf_query,
            "TF documento": tf_documento,
            "DF": df,
            "N": N,
            "IDF": idf,
            "Peso query": peso_query,
            "Peso documento": peso_documento,
            "Produto": contribuicao
        })

    norma_query = math.sqrt(
        soma_quadrados_query
    )

    norma_documento = (
        modelo_vetorial
        .document_norm
        .get(doc_dataset_id, 0.0)
    )

    score_calculado = (
        produto_escalar
        / (norma_query * norma_documento)
        if norma_query > 0 and norma_documento > 0
        else 0.0
    )

    score_modelo = modelo_vetorial.score(
        texto_query,
        doc_dataset_id,
        preprocessing_type=preprocessing_type
    )

    resumo = {
        "Categoria": categoria,
        "Doc.": doc_id,
        "N": N,
        "Norma query": norma_query,
        "Norma documento": norma_documento,
        "Produto escalar": produto_escalar,
        "Score recalculado": score_calculado,
        "Score do modelo": score_modelo
    }

    return resumo, linhas


def diagnostico_bm25(
    query_id,
    doc_id,
    categoria
):
    texto_query = consultas[query_id]["texto"]
    doc_dataset_id = documentos[doc_id]["id_dataset"]

    frequencias_query = Counter(
        preprocess(
            texto_query,
            preprocessing_type
        )
    )

    N = inverted_index.n_documents
    k1 = bm25.k1
    b = bm25.b
    dl = inverted_index.document_length[
        doc_dataset_id
    ]
    avgdl = inverted_index.avgdl

    score_calculado = 0.0
    linhas = []

    for termo, qtf in frequencias_query.items():
        postings = inverted_index.index.get(
            termo,
            {}
        )

        df = len(postings)
        tf = postings.get(doc_dataset_id, 0)

        if df > 0:
            idf = math.log(
                1 + (
                    (N - df + 0.5)
                    / (df + 0.5)
                )
            )
        else:
            idf = 0.0

        denominador = (
            tf
            + k1 * (
                1
                - b
                + b * (dl / avgdl)
            )
        )

        fator_tf = (
            (tf * (k1 + 1))
            / denominador
            if tf > 0 and denominador > 0
            else 0.0
        )

        contribuicao_unitaria = (
            idf * fator_tf
        )

        contribuicao_total = (
            qtf * contribuicao_unitaria
        )

        score_calculado += contribuicao_total

        linhas.append({
            "Categoria": categoria,
            "Doc.": doc_id,
            "Termo": termo,
            "QTF": qtf,
            "TF documento": tf,
            "DF": df,
            "N": N,
            "IDF BM25": idf,
            "k1": k1,
            "b": b,
            "DL": dl,
            "AvgDL": avgdl,
            "Fator TF": fator_tf,
            "Contribuição unitária": (
                contribuicao_unitaria
            ),
            "Contribuição total": (
                contribuicao_total
            )
        })

    score_modelo = bm25.score(
        texto_query,
        doc_dataset_id,
        preprocessing_type=preprocessing_type
    )

    resumo = {
        "Categoria": categoria,
        "Doc.": doc_id,
        "N": N,
        "k1": k1,
        "b": b,
        "DL": dl,
        "AvgDL": avgdl,
        "Score recalculado": score_calculado,
        "Score do modelo": score_modelo
    }

    return resumo, linhas


def exibir_calculos(
    query_id,
    nome_modelo,
    selecionados
):
    resumos = []
    detalhes = []

    ordem_documentos = {
        doc_id: ordem
        for ordem, (
            _,
            _,
            doc_id,
            _
        ) in enumerate(selecionados)
    }

    for categoria, _, doc_id, _ in selecionados:
        if nome_modelo == "Modelo Vetorial":
            resumo, termos = diagnostico_vetorial(
                query_id,
                doc_id,
                categoria
            )
        else:
            resumo, termos = diagnostico_bm25(
                query_id,
                doc_id,
                categoria
            )

        resumos.append(resumo)
        detalhes.extend(termos)

    tabela_resumo = pd.DataFrame(resumos)
    tabela_detalhes = pd.DataFrame(detalhes)

    tabela_resumo["_ordem"] = (
        tabela_resumo["Doc."]
        .map(ordem_documentos)
    )

    tabela_resumo = (
        tabela_resumo
        .sort_values("_ordem")
        .drop(columns="_ordem")
        .reset_index(drop=True)
    )

    print("\nRESUMO DO CÁLCULO DOS SCORES")

    if nome_modelo == "Modelo Vetorial":
        formatos_resumo = {
            "Norma query": "{:.6f}",
            "Norma documento": "{:.6f}",
            "Produto escalar": "{:.6f}",
            "Score recalculado": "{:.6f}",
            "Score do modelo": "{:.6f}"
        }
    else:
        formatos_resumo = {
            "k1": "{:.2f}",
            "b": "{:.2f}",
            "AvgDL": "{:.2f}",
            "Score recalculado": "{:.6f}",
            "Score do modelo": "{:.6f}"
        }

    estilo_resumo = estilizar(
        tabela_resumo,
        formatos_resumo
    )

    estilo_resumo = (
        estilo_resumo
        .bar(
            subset=["Score recalculado"],
            color="#7aa6c2"
        )
        .bar(
            subset=["Score do modelo"],
            color="#4c78a8"
        )
    )

    display(estilo_resumo)

    if nome_modelo == "Modelo Vetorial":
        coluna_contribuicao = "Produto"
        coluna_idf = "IDF"
        coluna_percentual = "Impacto no produto (%)"

        print(
            "\nFórmula: IDF = ln(N/DF); "
            "peso = (1 + ln(TF)) × IDF; "
            "score = produto escalar / "
            "(norma da query × norma do documento)."
        )

        formatos_detalhes = {
            "IDF": "{:.6f}",
            "Peso query": "{:.6f}",
            "Peso documento": "{:.6f}",
            "Produto": "{:.6f}",
            coluna_percentual: "{:.2f}%"
        }

    else:
        coluna_contribuicao = "Contribuição total"
        coluna_idf = "IDF BM25"
        coluna_percentual = "Impacto no score (%)"

        print(
            "\nFórmula: IDF = ln(1 + "
            "(N − DF + 0,5)/(DF + 0,5)); "
            "contribuição = IDF × "
            "[TF × (k1 + 1)] / "
            "[TF + k1 × "
            "(1 − b + b × DL/AvgDL)]."
        )

        formatos_detalhes = {
            "IDF BM25": "{:.6f}",
            "k1": "{:.2f}",
            "b": "{:.2f}",
            "AvgDL": "{:.2f}",
            "Fator TF": "{:.6f}",
            "Contribuição unitária": "{:.6f}",
            "Contribuição total": "{:.6f}",
            coluna_percentual: "{:.2f}%"
        }

    tabela_detalhes["Magnitude"] = (
        tabela_detalhes[
            coluna_contribuicao
        ].abs()
    )

    totais_por_documento = (
        tabela_detalhes
        .groupby("Doc.")["Magnitude"]
        .transform("sum")
    )

    tabela_detalhes[coluna_percentual] = [
        (
            100 * magnitude / total
            if total > 0
            else 0.0
        )
        for magnitude, total in zip(
            tabela_detalhes["Magnitude"],
            totais_por_documento
        )
    ]

    tabela_detalhes["Ordem do termo"] = (
        tabela_detalhes
        .groupby("Doc.")["Magnitude"]
        .rank(
            method="first",
            ascending=False
        )
        .astype(int)
    )

    tabela_detalhes["_ordem_documento"] = (
        tabela_detalhes["Doc."]
        .map(ordem_documentos)
    )

    tabela_detalhes = (
        tabela_detalhes
        .sort_values([
            "_ordem_documento",
            "Ordem do termo"
        ])
        .drop(columns=[
            "_ordem_documento",
            "Magnitude"
        ])
        .reset_index(drop=True)
    )

    colunas_iniciais = [
        "Categoria",
        "Doc.",
        "Ordem do termo",
        "Termo",
        coluna_percentual
    ]

    outras_colunas = [
        coluna
        for coluna in tabela_detalhes.columns
        if coluna not in colunas_iniciais
    ]

    tabela_detalhes = tabela_detalhes[
        colunas_iniciais + outras_colunas
    ]

    print(
        "\nCONTRIBUIÇÃO DOS TERMOS "
        "EM ORDEM DE IMPORTÂNCIA"
    )

    print(
        "Os termos estão ordenados da maior para "
        "a menor contribuição dentro de cada documento."
    )

    print(
        "\nBarras azuis: percentual explicado pelo termo."
        "\nBarras amarelas: magnitude do IDF."
        "\nBarras roxas: contribuição efetiva."
    )

    estilo_detalhes = estilizar(
        tabela_detalhes,
        formatos_detalhes
    )

    estilo_detalhes = (
        estilo_detalhes
        .bar(
            subset=[coluna_percentual],
            color="#5b9bd5",
            vmin=0,
            vmax=100
        )
        .bar(
            subset=[coluna_idf],
            color="#f2cf5b",
            vmin=0
        )
        .bar(
            subset=[coluna_contribuicao],
            color="#9c7bd8",
            vmin=0
        )
    )

    display(estilo_detalhes)


def exibir_modelo(query_id, nome_modelo):
    ranking = calcular_ranking(
        query_id,
        nome_modelo
    )

    falsos, omitidos = selecionar_erros(
        query_id,
        ranking
    )

    ids_falsos = {
        doc_id
        for _, doc_id, _ in falsos
    }

    ids_omitidos = {
        doc_id
        for _, doc_id, _ in omitidos
    }

    top10 = [
        (posicao, doc_id, score)
        for posicao, (doc_id, score) in enumerate(
            ranking[:TOP_K],
            start=1
        )
    ]

    print("\n" + "-" * 120)
    print(nome_modelo.upper())
    print("-" * 120)

    print("\nTOP-10")

    print(
        "Vermelho: dois primeiros documentos "
        f"não relevantes encontrados no Top-{TOPO}."
    )

    tabela_top10 = montar_tabela(
        query_id,
        top10,
        falsos=ids_falsos
    )

    display(
        estilizar(
            tabela_top10,
            {"Score": "{:.6f}"}
        )
    )

    print("\nRELEVANTES FORA DO TOP-10")

    print(
        "Verde: dois documentos relevantes mais "
        "bem posicionados depois do Top-10."
    )

    tabela_omitidos = montar_tabela(
        query_id,
        omitidos,
        omitidos=ids_omitidos
    )

    display(
        estilizar(
            tabela_omitidos,
            {"Score": "{:.6f}"}
        )
    )

    selecionados = [
        (
            "Falso positivo selecionado",
            posicao,
            doc_id,
            score
        )
        for posicao, doc_id, score in falsos
    ]

    selecionados += [
        (
            "Relevante fora do Top-10",
            posicao,
            doc_id,
            score
        )
        for posicao, doc_id, score in omitidos
    ]

    exibir_calculos(
        query_id,
        nome_modelo,
        selecionados
    )


def exibir_query(query_id):
    query_id = normalizar_id(query_id)
    texto_query = consultas[query_id]["texto"]

    print("\n\n" + "=" * 120)
    print(f"QUERY ORIGINAL {query_id}")
    print("=" * 120)

    print("\nTEXTO DA QUERY:")
    print(texto_query)

    print(
        f"\nDocumentos relevantes segundo os qrels: "
        f"{len(relevantes.get(query_id, set()))}"
    )

    print(
        "\nLegenda:"
        "\n- Vermelho: falso positivo selecionado."
        "\n- Verde: relevante fora do Top-10."
    )

    for nome_modelo in modelos:
        exibir_modelo(
            query_id,
            nome_modelo
        )


print("=" * 120)
print("9 — ANÁLISE DE ERROS")
print("=" * 120)

for query_id in QUERIES:
    exibir_query(query_id)

9 — ANÁLISE DE ERROS


QUERY ORIGINAL 57

TEXTO DA QUERY:
what are the significant steady and non-steady flow characteristics
which affect the flutter mechanism .

Documentos relevantes segundo os qrels: 14

Legenda:
- Vermelho: falso positivo selecionado.
- Verde: relevante fora do Top-10.

------------------------------------------------------------------------------------------------------------------------
MODELO VETORIAL
------------------------------------------------------------------------------------------------------------------------

TOP-10
Vermelho: dois primeiros documentos não relevantes encontrados no Top-5.


,Posição,Doc.,Score,Relevante,Categoria,Título,Texto
0,1,753,0.171582,Não,Falso positivo selecionado,development of a quasi-steady approach to flutter and correlation with kernel-function results .,development of a quasi-steady approach to flutter and correlation with kernel-function results . the quasi-steady approach to flutter utilizes experimental or theoretical steady-state aerodynamic data to arrive at increased understanding of the flutter mech...
1,2,1099,0.150651,Não,Falso positivo selecionado,a theoretical study of stagnation point ablation .,a theoretical study of stagnation point ablation . a simplified analysis is made of the shielding mechanism which reduces the stagnation-point heat transfer when ablation takes place at the surface . the most significant result of the analysis is that the e...
2,3,380,0.135794,Sim,,effect of quasi-steady air forces on incompressible bending-torsion flutter .,"effect of quasi-steady air forces on incompressible bending-torsion flutter . explicit solutions are obtained for the bending-torsion flutter of a two-dimensional airfoil in incompressible flow under the assumptions that the theodorsen function, c(k) is set..."
3,4,117,0.115749,Não,,the motion of a viscous liquid past a paraboloid .,the motion of a viscous liquid past a paraboloid . an approximate solution for the steady flow of incompressible viscous liquid past a paraboloid of revolution is described . an assumption is made for the form of the stokes stream function and substituted i...
4,5,1339,0.114794,Não,,calculation of flutter characteristics for finite-span swept or unswept wings at subsonic and sup...,calculation of flutter characteristics for finite-span swept or unswept wings at subsonic and supersonic speeds by a modified strip analysis . a method has been developed for calculating flutter characteristics of finite-span swept or unswept wings at subso...
5,6,444,0.114338,Não,,an approach to the flutter problem in real fluids .,"an approach to the flutter problem in real fluids . an approximate theory of airfoils in unsteady motion in a viscous fluid is proposed, in which viscous effects are accounted for by relaxing the kutta condition and replacing it by a relation derived from e..."
6,7,1181,0.113623,Não,,steady magnetohydrodynamic flow past a non-conducting wedge .,steady magnetohydrodynamic flow past a non-conducting wedge . this paper presents a study of the steady two-dimensional magnetohydrodynamic flow of an infinitely conducting fluid past a nonconducting wedge with nonaligned flow and magnetic field . the flows...
7,8,52,0.113242,Não,,"procedure for calculating flutter at high supersonic speed including camber deflections, and comp...","procedure for calculating flutter at high supersonic speed including camber deflections, and comparison with experimental results . a method which may be used at high supersonic mach numbers is described for calculating the flutter speed of wings having cam..."
8,9,1111,0.111271,Não,,some research on high speed flutter .,"some research on high speed flutter . paper presents brief discussions of many topics currently of interest in the flutter field . these include /a/ the sonic speed case, /b/ oscillating pressure field of propellers, /c/ wing flutter with various configurat..."
9,10,916,0.109796,Não,,the flow around oscillating low aspect ratio wings at transonic speeds .,"the flow around oscillating low aspect ratio wings at transonic speeds . when certain conditions are fulfilled for thickness ratio, aspect ratio, and reduced frequency for a three-dimensional wing, it can be shown that the partial differential equation for ..."



RELEVANTES FORA DO TOP-10
Verde: dois documentos relevantes mais bem posicionados depois do Top-10.


,Posição,Doc.,Score,Relevante,Categoria,Título,Texto
0,22,857,0.086679,Sim,Relevante fora do Top-10,experimental studies of flutter of buckled rectangular panels at mach numbers from 1. 2 to 3. 0 i...,experimental studies of flutter of buckled rectangular panels at mach numbers from 1. 2 to 3. 0 including effects of pressure differential and of panel width-length ratio . experimental panel flutter data have been obtained at mach numbers from 1.2 to 3.0 f...
1,24,593,0.084889,Sim,Relevante fora do Top-10,theoretical considerations of flutter at high mach number .,"theoretical considerations of flutter at high mach number . some of the theories for two-dimensional oscillatory air forces which may be applied in flutter calculations at high mach numbers are discussed . these include linear theory, van dyke's second-orde..."



RESUMO DO CÁLCULO DOS SCORES


...

,Posição,Doc.,Score,Relevante,Categoria,Título,Texto
0,17,704,11.485265,Sim,Relevante fora do Top-10,a systematic kernel function procedure for determining aerodynamic forces on oscillating or stead...,a systematic kernel function procedure for determining aerodynamic forces on oscillating or steady finite wings at subsonic speeds . a detailed description is given of a method of approximating solutions to the integral equation that relates oscillatory or ...
1,20,14,11.089742,Sim,Relevante fora do Top-10,piston theory - a new aerodynamic tool for the aeroelastician .,piston theory - a new aerodynamic tool for the aeroelastician . representative applications are described which illustrate the extent to which simplifications in the solutions of high-speed unsteady aeroelastic problems can be achieved through the use of ce...



RESUMO DO CÁLCULO DOS SCORES


,Categoria,Doc.,N,k1,b,DL,AvgDL,Score recalculado,Score do modelo
0,Falso positivo selecionado,753,1400,0.50,0.00,258,178.01,19.843326,19.843326
1,Falso positivo selecionado,1181,1400,0.50,0.00,155,178.01,14.872756,14.872756
2,Relevante fora do Top-10,704,1400,0.50,0.00,360,178.01,11.485265,11.485265
3,Relevante fora do Top-10,14,1400,0.50,0.00,386,178.01,11.089742,11.089742



Fórmula: IDF = ln(1 + (N − DF + 0,5)/(DF + 0,5)); contribuição = IDF × [TF × (k1 + 1)] / [TF + k1 × (1 − b + b × DL/AvgDL)].

CONTRIBUIÇÃO DOS TERMOS EM ORDEM DE IMPORTÂNCIA
Os termos estão ordenados da maior para a menor contribuição dentro de cada documento.

Barras azuis: percentual explicado pelo termo.
Barras amarelas: magnitude do IDF.
Barras roxas: contribuição efetiva.


,Categoria,Doc.,Ordem do termo,Termo,Impacto no score (%),QTF,TF documento,DF,N,IDF BM25,k1,b,DL,AvgDL,Fator TF,Contribuição unitária,Contribuição total
0,Falso positivo selecionado,753,1,steady,37.37%,2,10,104,1400,2.595754,0.50,0.00,258,178.01,1.428571,3.708221,7.416441
1,Falso positivo selecionado,753,2,flutter,23.22%,1,11,56,1400,3.210701,0.50,0.00,258,178.01,1.434783,4.606658,4.606658
2,Falso positivo selecionado,753,3,mechanism,20.19%,1,1,25,1400,4.006263,0.50,0.00,258,178.01,1.000000,4.006263,4.006263
3,Falso positivo selecionado,753,4,characteristics,11.08%,1,1,155,1400,2.198296,0.50,0.00,258,178.01,1.000000,2.198296,2.198296
4,Falso positivo selecionado,753,5,which,5.54%,1,3,595,1400,0.855540,0.50,0.00,258,178.01,1.285714,1.099980,1.099980
5,Falso positivo selecionado,753,6,are,2.18%,1,7,1028,1400,0.309085,0.50,0.00,258,178.01,1.400000,0.432719,0.432719
6,Falso positivo selecionado,753,7,and,0.32%,1,8,1339,1400,0.044890,0.50,0.00,258,178.01,1.411765,0.063374,0.063374
7,Falso positivo selecionado,753,8,the,0.10%,2,12,1391,1400,0.006804,0.50,0.00,258,178.01,1.440000,0.009798,0.019595
8,Falso positivo selecionado,753,9,what,0.00%,1,0,16,1400,4.441581,0.50,0.00,258,178.01,0.000000,0.000000,0.000000
9,Falso positivo selecionado,753,10,significant,0.00%,1,0,57,1400,3.193157,0.50,0.00,258,178.01,0.000000,0.000000,0.000000
